In [1]:
# libraries

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

In [ ]:
# load train

train = pd.read_csv("data/mnist_train.csv")
test = pd.read_csv("data/mnist_test.csv")

train.head()

,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# convert and check

train = np.array(train)
test = np.array(test)

for name, df in [("Train:", train), ("Test:", test)]:
    m, n = df.shape

    np.random.shuffle(df)

    print(name)
    print(type(df))
    print(m, n)

Train:
<class 'numpy.ndarray'>
60000 785
Test:
<class 'numpy.ndarray'>
10000 785


In [8]:
# train & test

X_train = train[:, 1:].T
X_train = X_train / 255.0 # scaling image pixel values

Y_train = train[:, 0]

x_val = test[:, 1:].T
x_val = x_val / 255.0 # scaling image pixel values

y_val = test[:, 0]

print(X_train.shape)
print(Y_train.shape)
print(x_val.shape)
print(y_val.shape)

(784, 60000)
(60000,)
(784, 10000)
(10000,)


In [ ]:
# functions

def initialize_parameters():
    w1 = np.random.rand(10, 784) - 0.5
    b1 = np.random.rand(10, 1) - 0.5

    w2 = np.random.rand(10, 10) - 0.5
    b2 = np.random.rand(10, 1) - 0.5

    return w1, b1, w2, b2


def ReLU(x):
    return np.maximum(x, 0)

def ReLU_derivative(x):
    return x > 0


def softmax(z):
    z -= np.max(z, axis=0, keepdims=True)
    return np.exp(z) / np.sum(np.exp(z), axis=0, keepdims=True)


def forward_propagation(w1, b1, w2, b2, x):
    z1 = w1 @ x + b1
    a1 = ReLU(z1)

    z2 = w2 @ a1 + b2
    a2 = softmax(z2) 

    return z1, a1, a2


def one_hot(y):
    one_hot_y = np.zeros((y.size, y.max() + 1))
    one_hot_y[np.arange(y.size), y] = 1

    return one_hot_y.T


def backward_propagation(w2, z1, a1, a2, x, y):
    one_hot_y = one_hot(y)

    m = x.shape[1]

    dz2 = a2 - one_hot_y
    dw2 = 1 / m * dz2 @ a1.T
    db2 = 1 / m * np.sum(dz2)

    dz1 = w2.T @ dz2 * ReLU_derivative(z1)
    dw1 = 1 / m * dz1 @ x.T
    db1 = 1 / m * np.sum(dz1)

    return dw1, db1, dw2, db2


def update_parameters(w1, b1, w2, b2, dw1, db1, dw2, db2, lr):
    w1 = w1 - lr * dw1
    b1 = b1 - lr * db1
    w2 = w2 - lr * dw2
    b2 = b2 - lr * db2
    
    return w1, b1, w2, b2


def pred(a2):
    return np.argmax(a2, axis=0)


def get_acc(predictions, y):
    return np.sum(predictions == y) / y.size


def gradient_descent(x, y, lr, ni):
    w1, b1, w2, b2 = initialize_parameters()

    for i in range(ni):
        z1, a1, a2 = forward_propagation(w1, b1, w2, b2, x)

        dw1, db1, dw2, db2 = backward_propagation(w2, z1, a1, a2, x, y)

        w1, b1, w2, b2 = update_parameters(w1, b1, w2, b2, dw1, db1, dw2, db2, lr)

        if (i%20)==0:
            print("Number of iterations:", i)
            print("Accuracy:", get_acc(pred(a2), y))

    return w1, b1, w2, b2